# B4 — #128'den damıtma: kendi modelimiz 0.2 altına

**Amaç:** Kendi HNeRV modelimizi (`b3_ent5b`, 0.240, latent 36) öğretmen olarak #128'i (CPU 0.187961) kullanarak daha iyi bir havzaya taşımak; sonra bildiğimiz tarifle (7 bit QAT + entropi cezası + canlı final) sıkıştırmak.

**Hipotez (tez):** Bizimle #95/#128 arasındaki dist farkı (0.104 vs 0.070) kapasite değil, eğitimin ulaştığı havza. Mimari aynı boyda (aynı kanallar; latent 36 ≥ 28), yani öğrenci öğretmene yetişebilmeli.

**Sıra**
| Bölüm | Ne | Süre (A100) |
|---|---|---|
| 1 | Kurulum: Drive, challenge reposu, #128 arşivi | ~5 dk |
| 2 | Değerlendirici, PyAV hedefleri, skor zinciri | ~5 dk |
| 3 | Öğretmen: #128'in decoder çıktıları (384×512) | ~2 dk |
| 4 | Öğrenci ve araçlar; `ent5b` sağlaması (GT kontrolü de burada) | ~2 dk |
| 5 | **Faz 1 — damıtma** (FP32, değerlendiricisiz, L1) | ~30–45 dk |
| 6 | **Faz 2a — 7 bit QAT** (görev kaybı, λ=0, lr 1e-4) | ~2.5 sa |
| 7 | **Faz 2b — entropi cezası** (λ=5, lr 3e-5, canlı final) | ~1 sa |
| 8 | Özet tablo | dk |

**Karar noktaları**
- Bölüm 4: `ent5b`'nin yeni (PyAV) hedeflerle toplamı eski 0.240'a yakın mı? Uzaksa fark = eski hedeflerin GT hatası (bilgi; yine devam).
- Faz 1 sonu: FP32 dist, `ent5b`'nin FP32 dist'inden **≥ 0.01** aşağıda mı? Değilse damıtma havzayı değiştirmedi → dur (kayıp < 1 saat).
- Faz 2 sonu: 7 bit toplam < 0.2 mi?

**Kurallar (değişmedi):** yalnız SON satırı okunur · her koşuya yeni etiket · tek değişken · canlı model LR=0'da ölçülür · Colab koparsa aynı hücreyi aynı etiketle yeniden çalıştır (kaldığı yerden devam eder).

Öğretmen #128 (MIT, PR #128 / #112 / #101 / #95 zinciri) — tezde ve olası gönderimde atıf yapılır.

## 1. Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
sh = lambda cmd: subprocess.run(cmd, shell=True, check=True)
SUB128 = "/content/cc/submissions/rhnerv_latent_polish"
if not os.path.exists("/content/cc"):
    sh("apt-get -qq install -y git-lfs > /dev/null && git lfs install > /dev/null")
    sh("git clone -q https://github.com/commaai/comma_video_compression_challenge.git /content/cc")
    sh("cd /content/cc && git lfs pull && git fetch -q origin pull/128/head:pr128 && git checkout -q pr128 && git lfs pull")
if not os.path.exists(f"{SUB128}/archive.zip"):
    sh(f"curl -sL -o {SUB128}/archive.zip https://github.com/a12dongithub/comma_video_compression_challenge/releases/download/rhnerv-latent-polish-20260709/archive.zip")
sh("pip -q install av einops timm safetensors segmentation-models-pytorch brotli constriction pyppmd")
print(os.path.getsize(f"{SUB128}/archive.zip"), "bayt (176531 olmalı)")

In [ ]:
import sys, math, copy, zipfile, importlib
from pathlib import Path
import numpy as np
import torch, einops
import torch.nn as nn
import torch.nn.functional as F

DRIVE = Path("/content/drive/MyDrive/deneme")
OUT_DIR = DRIVE / "b4"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG = OUT_DIR / "b4_log.txt"
ENT5B = DRIVE / "b3" / "b3_ent5b.pt"


def log(msg):
    print(msg)
    with open(LOG, "a") as f:
        f.write(msg + "\n")


dev = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = True   # training speed; score() turns it off

## 2. Değerlendirici, hedefler, skor zinciri
Hedefler resmi CPU yolundan (PyAV + `yuv420_to_rgb`) üretilmiş `hedefler_pyav.pt`; yoksa burada yeniden üretilir. Skor zinciri resmi değerlendirmeyi kopyalar: decoder çıktısı (384×512) → bikübik 874×1164 → clamp → yuvarlama → değerlendiricinin bilineer küçültmesi.

In [ ]:
sys.path.insert(0, "/content/cc")
from frame_utils import AVVideoDataset
from modules import DistortionNet, segnet_sd_path, posenet_sd_path

net = DistortionNet().eval().to(dev)
net.load_state_dicts(posenet_sd_path, segnet_sd_path, dev)
net.requires_grad_(False)

HEDEF = DRIVE / "p128" / "hedefler_pyav.pt"
if not HEDEF.exists():
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    ds = AVVideoDataset(["0.mkv"], data_dir=Path("/content/cc/videos"), batch_size=16, device=torch.device("cpu"))
    ds.prepare_data()
    seg, pose = [], []
    with torch.inference_mode():
        for _, _, batch in ds:
            p_out, s_out = net(batch.to(dev))
            seg.append(s_out.argmax(1).to(torch.uint8).cpu())
            pose.append(p_out["pose"][:, :6].float().cpu())
    HEDEF.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"seg_class": torch.cat(seg), "pose": torch.cat(pose)}, HEDEF)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = True

hed = torch.load(HEDEF)
seg_targets = hed["seg_class"].to(dev).long()
pose_targets = hed["pose"].to(dev).float()
NUM_PAIRS = seg_targets.shape[0]
PAIRS = torch.arange(NUM_PAIRS, device=dev)
print("hedefler:", tuple(seg_targets.shape), tuple(pose_targets.shape))

In [ ]:
def yuv6(rgb):   # differentiable copy: the official rgb_to_yuv6 is no_grad and clamps in place, so pose gradients would die
    R, G, B = rgb[:, 0], rgb[:, 1], rgb[:, 2]
    Y = (0.299 * R + 0.587 * G + 0.114 * B).clamp(0, 255)
    U = ((B - Y) / 1.772 + 128).clamp(0, 255)
    V = ((R - Y) / 1.402 + 128).clamp(0, 255)
    pool = lambda c: 0.25 * (c[:, 0::2, 0::2] + c[:, 1::2, 0::2] + c[:, 0::2, 1::2] + c[:, 1::2, 1::2])
    return torch.stack([Y[:, 0::2, 0::2], Y[:, 1::2, 0::2], Y[:, 0::2, 1::2], Y[:, 1::2, 1::2], pool(U), pool(V)], 1)


def inflate(native):
    b = native.shape[0]
    up = F.interpolate(native.flatten(0, 1), size=(874, 1164), mode="bicubic", align_corners=False).clamp(0, 255)
    up = up + (up.round() - up).detach()   # STE: the archive ships uint8 frames
    return up.view(b, 2, 3, 874, 1164)


def seg_logits(full):
    return net.segnet(F.interpolate(full[:, 1], size=(384, 512), mode="bilinear"))


def pose_pred(full):
    x = F.interpolate(full.flatten(0, 1), size=(384, 512), mode="bilinear")
    return net.posenet(einops.rearrange(yuv6(x), "(b t) c h w -> b (t c) h w", t=2))["pose"][:, :6]


L7 = dict(tau=0.3, thr=1.0, mult=4.0)


def l7_loss(logits, truth):
    correct = logits.gather(1, truth.unsqueeze(1))
    others = logits.clone()
    others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
    margin = correct - others.amax(1, keepdim=True)
    per_pixel = L7["tau"] * F.softplus(-margin / L7["tau"])
    with torch.no_grad():
        w = 1.0 + L7["mult"] * (margin < L7["thr"]).float()
        w = w / w.mean()
    return (per_pixel * w).mean()


def task_loss(native, ids):
    full = inflate(native)
    seg_l = 100 * l7_loss(seg_logits(full), seg_targets[ids])
    pose_l = torch.sqrt(10 * F.mse_loss(pose_pred(full), pose_targets[ids]) + 1e-6)
    return seg_l + pose_l, seg_l.item(), pose_l.item()


@torch.no_grad()
def score(render, bs=16):
    old = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False   # match the official FP32 numerics
    wrong, pose_sq = 0, 0.0
    for s in range(0, NUM_PAIRS, bs):
        ids = PAIRS[s:s + bs]
        full = inflate(render(ids))
        wrong += int((seg_logits(full).argmax(1) != seg_targets[ids]).sum())
        pose_sq += float((pose_pred(full) - pose_targets[ids]).square().mean(1).sum())
    torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = old
    seg = 100 * wrong / (NUM_PAIRS * 384 * 512)
    pose = (10 * pose_sq / NUM_PAIRS) ** 0.5
    return {"dist": seg + pose, "seg": seg, "pose": pose}

## 3. Öğretmen: #128'in decoder çıktıları
#128'in arşivi kendi açma koduyla (MIT) okunur; decoder 600 pair için 384×512 çıktı üretir (büyütme ve #98/FEC6 son işlemleri öncesi). Bu kareler Faz 1'in hedefi. Sağlama: öğretmenin bizim zincirimizdeki dist'i, #128'in resmi dist'ine (0.0704) yakın ama biraz kötü çıkmalı, çünkü kanal bias'ları ve FEC6 seçici burada yok.

In [ ]:
sys.path.insert(0, SUB128)
t128 = importlib.import_module("inflate")   # #128's own decode functions
member = zipfile.ZipFile(f"{SUB128}/archive.zip").read("x")
sd128, lat128, _, _ = t128.parse_member(member)
teacher = t128.HNeRVDecoder(latent_dim=t128.LATENT_DIM, base_channels=t128.BASE_CHANNELS,
                            eval_size=t128.EVAL_SIZE).to(dev).eval()
teacher.load_state_dict(sd128)
lat128 = lat128.to(dev).float()

torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
with torch.no_grad():
    TEACH = torch.cat([teacher(lat128[s:s + 16]).half() for s in range(0, NUM_PAIRS, 16)])
torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = True
r = score(lambda ids: TEACH[ids].float())
log(f"[öğretmen] #128 decoder, bizim zincir | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) | resmi #128 dist 0.0704")
del teacher

## 4. Öğrenci ve araçlar
Öğrenci = bizim B3 modelimiz (aynı decoder mimarisi, latent 36). Araçlar B3 notebook'undakilerle aynı: kanal başına (Conv) / tensör başına (Linear) simetrik nicemleme, STE'li QAT, ideal entropi ile KB, türevlenebilir yumuşak KB (entropi cezası), EMA, atomik durum kaydı.

In [ ]:
class HNeRVDecoder(nn.Module):
    def __init__(self, latent_dim=36, base_channels=36):
        super().__init__()
        c = base_channels
        self.channels = [c, c, c, int(c * 0.75), int(c * 0.58), int(c * 0.5), int(c * 0.5)]
        self.stem = nn.Linear(latent_dim, self.channels[0] * 6 * 8)
        self.blocks, self.skips = nn.ModuleList(), nn.ModuleList()
        for i in range(6):
            cin, cout = self.channels[i], self.channels[i + 1]
            self.blocks.append(nn.Conv2d(cin, cout * 4, 3, padding=1))
            self.skips.append(nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity())
        self.ps = nn.PixelShuffle(2)
        f = self.channels[-1]
        self.refine = nn.Sequential(nn.Conv2d(f, f // 2, 3, padding=2, dilation=2), nn.Conv2d(f // 2, f, 3, padding=1))
        self.rgb_0 = nn.Conv2d(f, 3, 3, padding=1)
        self.rgb_1 = nn.Conv2d(f, 3, 3, padding=1)

    def forward(self, z):
        x = torch.sin(self.stem(z).view(len(z), self.channels[0], 6, 8))
        for block, skip in zip(self.blocks, self.skips):
            identity = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            x = torch.sin(self.ps(block(x)) + skip(identity))
        x = x + 0.1 * torch.sin(self.refine(x))
        return torch.stack([torch.sigmoid(self.rgb_0(x)) * 255, torch.sigmoid(self.rgb_1(x)) * 255], dim=1)


class B3(nn.Module):
    def __init__(self, n_pairs, latent_dim=36):
        super().__init__()
        self.codes = nn.Parameter(torch.zeros(n_pairs, latent_dim))
        self.dec = HNeRVDecoder(latent_dim)

    def forward(self, ids):
        return self.dec(self.codes[ids])


def kaynak_yukle(yol):
    sd = torch.load(yol, map_location="cpu", weights_only=False)["model"]
    m = B3(NUM_PAIRS, sd["codes"].shape[1]).to(dev)
    m.load_state_dict(sd)
    return m

In [ ]:
rate_of_kb = lambda kb: 25 * kb * 1e3 / 37545489


def qmax_of(bits):
    return int(round(2 ** (bits - 1) - 1))


def katmanlar(m):
    return [(ad, mod) for ad, mod in m.named_modules() if isinstance(mod, (nn.Conv2d, nn.Linear))]


def _olcek(w, qmax):
    if w.dim() == 4:
        return w.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
    return w.abs().amax().clamp_min(1e-12) / qmax


@torch.no_grad()
def quant_net_(m, bits):
    qmax = qmax_of(bits)
    for _, mod in katmanlar(m):
        s = _olcek(mod.weight, qmax)
        mod.weight.copy_(torch.round(mod.weight / s).clamp(-qmax, qmax) * s)


def qat_ac(m, bits):
    qmax, asil = qmax_of(bits), {}
    for ad, mod in katmanlar(m):
        w = mod.weight.data
        asil[ad] = w
        s = _olcek(w, qmax)
        mod.weight.data = torch.round(w / s).clamp(-qmax, qmax) * s
    return asil


def qat_kapat(m, asil):
    for ad, mod in katmanlar(m):
        mod.weight.data = asil[ad]


def entropy_kb(q):
    _, n = torch.unique(q, return_counts=True)
    p = n.double() / q.numel()
    return float(-(n * p.log2()).sum()) / 8 / 1e3


@torch.no_grad()
def decoder_kb(dec, bits):
    qmax, kb = qmax_of(bits), 0.0
    for _, mod in katmanlar(dec):
        s = _olcek(mod.weight, qmax)
        kb += entropy_kb(torch.round(mod.weight / s).clamp(-qmax, qmax)) + 2 * s.numel() / 1e3
        if mod.bias is not None:
            kb += 2 * mod.bias.numel() / 1e3
    return kb


def kod_kb(codes, bits=8):
    qmax = qmax_of(bits)
    s = codes.abs().amax(dim=0, keepdim=True).clamp_min(1e-12) / qmax
    q = torch.round(codes / s).clamp(-qmax, qmax)
    return q * s, entropy_kb(q) + 2 * s.numel() / 1e3


@torch.no_grad()
def olc(m, bits=7, kod_bits=8):
    em = copy.deepcopy(m)
    quant_net_(em.dec, bits)
    q, kkb = kod_kb(em.codes.data, kod_bits)
    em.codes.data = q
    r = score(em)
    dkb = decoder_kb(em.dec, bits)
    del em
    torch.cuda.empty_cache()
    return r | {"dec_kb": dkb, "kod_kb": kkb, "toplam": r["dist"] + rate_of_kb(dkb + kkb)}


def yumusak_kb(dec, bits):
    qmax, kb = qmax_of(bits), 0.0
    for _, mod in katmanlar(dec):
        s = _olcek(mod.weight.detach(), qmax)   # scale held fixed: otherwise the penalty could cheat by moving the max weight
        x = (mod.weight / s).clamp(-qmax, qmax) + qmax
        lo = x.detach().floor().clamp(max=2 * qmax - 1)
        fr = (x - lo).flatten()
        lo = lo.long().flatten()
        sayim = torch.zeros(2 * qmax + 1, device=x.device).index_add(0, lo, 1 - fr).index_add(0, lo + 1, fr)
        kb = kb - (sayim * torch.log2(sayim / fr.numel() + 1e-12)).sum() / 8 / 1e3
    return kb


class EMA:
    def __init__(self, model, ufuk_adim):
        self.decay = 1 - 1 / max(1, ufuk_adim)
        self.golge = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.n = 0

    @torch.no_grad()
    def guncelle(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        for k, v in model.state_dict().items():
            self.golge[k].lerp_(v, 1 - d)

    def kopya(self, sablon):
        m = copy.deepcopy(sablon)
        m.load_state_dict(self.golge)
        return m


def durum_kaydet(yol, **durum):
    gecici = yol.with_suffix(".tmp")
    torch.save(durum, gecici)
    os.replace(gecici, yol)   # a disconnect mid-write must leave either the old or the new file, never half of one


def carpan_fn(epochs, adim, warm):
    return lambda s: min(1, (s + 1) / (warm * adim)) * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim))))

### 4.1 `ent5b` sağlaması
İki sayı: FP32 dist (Faz 1'in kıyas noktası) ve 7 bit toplam. Eski notebook'ta toplam **0.240** idi; fark, eski hedeflerin resmi PyAV yolundan ne kadar saptığını gösterir (GT kontrolü).

In [ ]:
m = kaynak_yukle(ENT5B)
r = score(m)
q = olc(m, 7)
log(f"[ent5b] FP32 dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f})")
log(f"[ent5b] 7 bit dist {q['dist']:.4f} | decoder {q['dec_kb']:.1f} + latent {q['kod_kb']:.1f} KB | TOPLAM {q['toplam']:.3f} (eski hedeflerle 0.240)")
ENT5B_FP32 = r["dist"]
del m; torch.cuda.empty_cache()

## 5. Faz 1 — damıtma (FP32, değerlendiricisiz)
Öğrenci `ent5b`'den başlar; kayıp, öğrencinin 384×512 çıktısı ile öğretmenin çıktısı arasındaki L1 (0–255 ölçeği). QAT yok, SegNet/PoseNet yok → epoch'lar ucuz. LR #95'in FP32 fazının aralığında (yüksek), çünkü amaç havza değiştirmek. Her `eval_every` epoch'ta gerçek FP32 dist ölçülür (L1'in düşmesi tek başına yetmez).

**Karar:** SON satırındaki FP32 dist ≤ `ENT5B_FP32 − 0.01` → Faz 2'ye geç. Değilse dur.

In [ ]:
def damit(init, tag, epochs=2000, batch=20, lr_dec=5e-4, lr_kod=5e-3, warm=20, eval_every=200):
    durum_yol = OUT_DIR / f"b4_{tag}_durum.pt"
    m = kaynak_yukle(init)
    opt = torch.optim.Adam([{"params": m.dec.parameters(), "lr": lr_dec}, {"params": [m.codes], "lr": lr_kod}])
    adim = math.ceil(NUM_PAIRS / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, carpan_fn(epochs, adim, warm))
    bas = 1
    if durum_yol.exists():
        d = torch.load(durum_yol, map_location=dev, weights_only=False)
        m.load_state_dict(d["model"]); opt.load_state_dict(d["opt"]); sched.load_state_dict(d["sched"])
        bas = d["epoch"] + 1
        log(f"[{tag}] devam: E{bas}'den")
    else:
        r = score(m)
        log(f"[{tag}] başla: {init} | lr {lr_dec:g}/{lr_kod:g} | epoch {epochs}, batch {batch} | E0 FP32 dist {r['dist']:.4f}")

    top, n = 0.0, 0
    for ep in range(bas, epochs + 1):
        for ids in torch.randperm(NUM_PAIRS, device=dev).split(batch):
            loss = (m(ids) - TEACH[ids].float()).abs().mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
            top += loss.item(); n += 1
        if ep % eval_every and ep != epochs:
            continue
        r = score(m)
        log(f"[{tag}] E{ep:04d} | lr {sched.get_last_lr()[0]:.1e} | L1 {top / n:.3f} | "
            f"FP32 dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f})")
        top, n = 0.0, 0
        durum_kaydet(durum_yol, model=m.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=ep)

    torch.save({"model": m.state_dict()}, OUT_DIR / f"b4_{tag}.pt")
    durum_yol.unlink()
    log(f"[{tag}] SON | FP32 dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) | ent5b FP32 {ENT5B_FP32:.4f}")
    return r

In [ ]:
_ = damit(ENT5B, "dam1")

## 6. Faz 2a — 7 bit QAT, görev kaybı (λ=0)
Damıtılmış FP32 modeli 7 bit ızgaraya taşır. LR 1e-4: q7 bu LR'de 7 bitte sorunsuz çalışmıştı. Entropi cezası bu fazda **kapalı**, çünkü doğrusal çekirdekli ceza yüksek LR'de kararsızdı (hlr). Fonksiyon B3'teki `b3_egit`'in aynısı: canlı + EMA ölçümü, en iyi toplam, devam altyapısı.

In [ ]:
def egit(init, tag, lr_dec=1e-4, lr_kod=1e-3, epochs=1500, batch=20, bits=7, kod_bits=8,
         warm=10, ema_ep=25, eval_every=100, lam=0.0):
    durum_yol = OUT_DIR / f"b4_{tag}_durum.pt"
    m = kaynak_yukle(init)
    opt = torch.optim.Adam([{"params": m.dec.parameters(), "lr": lr_dec}, {"params": [m.codes], "lr": lr_kod}])
    adim = math.ceil(NUM_PAIRS / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, carpan_fn(epochs, adim, warm))
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    en_iyi = {"toplam": float("inf")}
    bas = 1

    if durum_yol.exists():
        d = torch.load(durum_yol, map_location=dev, weights_only=False)
        m.load_state_dict(d["model"]); opt.load_state_dict(d["opt"]); sched.load_state_dict(d["sched"])
        if ema:
            ema.golge, ema.n = d["ema_golge"], d["ema_n"]
        en_iyi, bas = d["en_iyi"], d["epoch"] + 1
        log(f"[{tag}] devam: E{bas}'den | şimdiye kadar en iyi {en_iyi['toplam']:.3f} (E{en_iyi['epoch']}, {en_iyi['kaynak']})")
    else:
        log(f"[{tag}] başla: {init} | lr {lr_dec:g}/{lr_kod:g} | {bits} bit, latent {kod_bits} | "
            f"epoch {epochs}, batch {batch}, ema {ema_ep}, λ {lam}")

    top = seg_t = pose_t = kb_t = 0.0
    n = 0
    for ep in range(bas, epochs + 1):
        for ids in torch.randperm(NUM_PAIRS, device=dev).split(batch):
            opt.zero_grad(set_to_none=True)
            if lam:
                kb_y = yumusak_kb(m.dec, bits)
                (lam * rate_of_kb(kb_y)).backward()   # on FP32 weights, before qat_ac swaps them for rounded ones
                kb_t += kb_y.item()
            asil = qat_ac(m.dec, bits)
            loss, s_l, p_l = task_loss(m(ids), ids)
            loss.backward()
            qat_kapat(m.dec, asil)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            sched.step()
            if ema:
                ema.guncelle(m)
            top += loss.item(); seg_t += s_l; pose_t += p_l; n += 1

        if ep % eval_every and ep != epochs:
            continue

        log(f"[{tag}] E{ep:04d} | lr {sched.get_last_lr()[0]:.1e} | "
            f"eğitim kaybı {top / n:.4f} (seg {seg_t / n:.4f}, pose {pose_t / n:.4f})"
            + (f" | kb~ {kb_t / n:.1f}" if lam else ""))
        top = seg_t = pose_t = kb_t = 0.0
        n = 0

        adaylar = [("canlı", m)] + ([("ema", ema.kopya(m))] if ema else [])
        for kaynak, aday in adaylar:
            r = olc(aday, bits, kod_bits)
            log(f"[{tag}]        {kaynak:5s} | {bits} bit dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) "
                f"| decoder {r['dec_kb']:.1f} + latent {r['kod_kb']:.1f} KB | toplam {r['toplam']:.3f}")
            if r["toplam"] < en_iyi["toplam"]:
                en_iyi = r | {"epoch": ep, "kaynak": kaynak, "sd": copy.deepcopy(aday.state_dict())}
        log(f"[{tag}]        en iyi {en_iyi['toplam']:.3f} (E{en_iyi['epoch']}, {en_iyi['kaynak']})")

        durum_kaydet(durum_yol, model=m.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                     ema_golge=ema.golge if ema else None, ema_n=ema.n if ema else 0,
                     en_iyi=en_iyi, epoch=ep)

    log(f"[{tag}] SON | en iyi E{en_iyi['epoch']} ({en_iyi['kaynak']}) | {bits} bit dist {en_iyi['dist']:.4f} "
        f"(seg {en_iyi['seg']:.4f}, pose {en_iyi['pose']:.4f}) | decoder {en_iyi['dec_kb']:.1f} + "
        f"latent {en_iyi['kod_kb']:.1f} KB | TOPLAM {en_iyi['toplam']:.3f}")
    torch.save({"model": en_iyi["sd"], "bits": bits, "kod_bits": kod_bits}, OUT_DIR / f"b4_{tag}.pt")
    durum_yol.unlink()
    return en_iyi

In [ ]:
_ = egit(OUT_DIR / "b4_dam1.pt", "q7a", lr_dec=1e-4, lr_kod=1e-3, epochs=1500, bits=7, lam=0.0)

## 7. Faz 2b — entropi cezası (λ=5), düşük LR, canlı final
B3'te 0.248 → 0.244 → 0.240 zincirini veren tarif. Tur kazancı ≥ 0.003 ise aynı tarifle bir tur daha (`q7c`, başlangıç `b4_q7b.pt`).

In [ ]:
_ = egit(OUT_DIR / "b4_q7a.pt", "q7b", lr_dec=3e-5, lr_kod=3e-4, epochs=600, bits=7, eval_every=50, lam=5.0)

## 8. Özet

In [ ]:
satirlar = [("ent5b (başlangıç)", ENT5B)] + [(t, OUT_DIR / f"b4_{t}.pt") for t in ("q7a", "q7b", "q7c")]
for ad, yol in satirlar:
    if not Path(yol).exists():
        continue
    r = olc(kaynak_yukle(yol), 7)
    log(f"[özet] {ad:18s} | 7 bit dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) | "
        f"{r['dec_kb'] + r['kod_kb']:.1f} KB | TOPLAM {r['toplam']:.3f}")
log("[özet] kıyas: #128 resmi CPU 0.187961 · hedef < 0.2")

**Sonraki adım (bu notebook dışında):** en iyi checkpoint için gerçek arşiv yazıcısı + `inflate.sh` (gönderim kodu, Hamza) ve resmi `evaluate.sh --device cpu` doğrulaması. GPU'daki `olc` değeri resmi CPU eksenine ±0.002 içinde beklenir (#128 için bu notebook'un zinciri birebir tuttu).